# Analyse: ästhetische Qualität


In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision pillow numpy pandas tqdm transformers huggingface_hub


In [8]:
from pathlib import Path
import os
import sys
import types
import importlib.util

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from transformers import AutoProcessor
from huggingface_hub import snapshot_download

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())



2.7.1+cpu
CUDA available: False


In [5]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '08_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_aesthetic_quality.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60

REPO_ID = 'shunk031/aesthetics-predictor-v1-vit-base-patch32'

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')

df = pd.read_csv(COMBINED_CSV)
if 'influencer_type' not in df.columns and 'source' in df.columns:
    df['influencer_type'] = df['source']


Running on device: cpu


In [6]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'

def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)

def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [9]:
def load_aesthetic_predictor(repo_id: str):
    repo_dir = snapshot_download(repo_id)

    # Temporären Paketkontext für relative Importe im Modellcode setzen
    package_name = 'aesthetics_predictor_repo'
    if package_name not in sys.modules:
        sys.modules[package_name] = types.ModuleType(package_name)
        sys.modules[package_name].__path__ = [repo_dir]

    modeling_file = None
    for fname in os.listdir(repo_dir):
        if fname.startswith('modeling_') and fname.endswith('.py'):
            modeling_file = os.path.join(repo_dir, fname)
            break
    if modeling_file is None:
        raise FileNotFoundError(f'No modeling_*.py found in {repo_dir}')

    spec = importlib.util.spec_from_file_location(f'{package_name}.modeling', modeling_file)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f'{package_name}.modeling'] = module
    spec.loader.exec_module(module)

    predictor_cls = None
    for name, obj in module.__dict__.items():
        if isinstance(obj, type) and 'Predictor' in name and 'Config' not in name:
            predictor_cls = obj
            break
    if predictor_cls is None:
        raise ValueError('No predictor class found in remote modeling module.')

    processor = AutoProcessor.from_pretrained(repo_dir, trust_remote_code=True)
    model = predictor_cls.from_pretrained(repo_dir)
    model.to(device).eval()
    return model, processor


model, processor = load_aesthetic_predictor(REPO_ID)

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []
    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []
    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan
    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames
    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]

def get_aesthetic_score(image_path: Path):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        if hasattr(outputs, 'logits') and outputs.logits is not None:
            score_tensor = outputs.logits
        elif torch.is_tensor(outputs):
            score_tensor = outputs
        elif isinstance(outputs, (tuple, list)) and len(outputs) > 0 and torch.is_tensor(outputs[0]):
            score_tensor = outputs[0]
        else:
            return np.nan
        return float(score_tensor.reshape(-1)[0].item())
    except Exception:
        return np.nan



Fetching 12 files: 100%|██████████| 12/12 [00:00<?, ?it/s]


In [10]:
video_scores = []
video_scored_frames = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    vid = get_video_id(row)
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

    scores = []
    for fp in frame_paths:
        score = get_aesthetic_score(fp)
        if not np.isnan(score):
            scores.append(score)

    if scores:
        video_scores.append(float(np.mean(scores)))
        video_scored_frames.append(len(scores))
    else:
        video_scores.append(np.nan)
        video_scored_frames.append(0)


Processing videos: 100%|██████████| 500/500 [21:34<00:00,  2.59s/it]


In [11]:
df['aesthetic_quality_score'] = video_scores
df['aesthetic_quality_scored_frames'] = video_scored_frames

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
df[['aesthetic_quality_score', 'aesthetic_quality_scored_frames']].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\08_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_aesthetic_quality.csv


,aesthetic_quality_score,aesthetic_quality_scored_frames
0,5.380172,7
1,4.282832,10
2,2.873493,8
3,5.395767,8
4,4.784345,8
5,6.444030,8
6,6.022556,8
7,5.511352,8
8,5.927886,6
9,5.336604,8
